<a href="https://colab.research.google.com/github/rnwjdgus03/NLP_05-Team-Project-3/blob/Category-Extension/article_combined_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 기사 통합 추출 파이프라인 (Colab) — is_claim 판별 + 컬럼 추출 동시 처리

`extract_article_claims_hcx.py` 하나로 기사 전체를 처리합니다.

```
articles.csv (기사 본문 전체)
   ↓ extract_article_claims_hcx.py
   기사 전체를 HCX에 통째로 넣어
     1) 검증 가능한 claim 문장을 찾고 (is_claim 판별)
     2) 동시에 그 문장의 수치/통계 컬럼을 추출              ← API 호출
최종 데이터셋 (hcx_article_extracted.csv)
```

문장 단위로 나눠서 처리하는 이전 버전(`article_to_claims.py` → `is_claim_filter_hcx.py`
→ `filter_true_claims.py` → `extract_hcx.py`)과 달리, 기사 전체 맥락(제목·다른 문단 포함)을
보고 판단하기 때문에 시점(period)처럼 문장 하나만으로는 알 수 없는 정보를 더 잘 채울 수
있습니다. 대신 기사(또는 chunk) 하나당 API 호출이 크고 무거우니, 처음엔 `LIMIT`을 작게
두고 결과를 확인한 뒤 늘리세요. 중단 후 재실행하면 이미 처리한 기사(article_id)는 건너뛰고
이어서 처리됩니다.


## 0. 준비물

- 스크립트 2개: `extract_hcx.py`, `extract_article_claims_hcx.py`
  (`extract_article_claims_hcx.py`가 `extract_hcx.py`의 검증/보정 함수를 그대로 불러써서
  두 파일 모두 필요합니다.)
- 기사 원문 CSV: `article_id, title, date, url, body` 칼럼 (body = 기사 본문 전체 텍스트)
- Colab 보안 비밀(Secrets)에 `CLOVA_API_KEY` 등록
  - 왼쪽 사이드바 🔑(보안 비밀) 아이콘 → "새 보안 비밀 추가"
  - 이름: `CLOVA_API_KEY`, 값: 발급받은 키, **"노트북 액세스" 토글 켜기**


In [ ]:
# 필요한 패키지 설치
!pip install -q requests python-dotenv


In [ ]:
!python is_claim_filter_hcx.py --input all_sentences_51_100.csv --output is_claim_result.csv
# python filter_true_claims.py --input is_claim_result.csv --output claims_filtered.csv
# python extract_hcx.py --input claims_filtered.csv --output hcx_extracted.csv

[B0001-s000] False (hcx) 현재 문장은 특정 전문가의 의견 및 예측을 담고 있으며, 이는 KOSIS
[B0001-s001] True (hcx) 현재 문장은 올해 서울, 경기도 및 지방의 집값 상승 및 하락 예측을 포
[B0001-s002] False (hcx) 탄핵 정국이 환율 급등, 공사비 인상, 분양가 상승 등의 영향을 미칠 것
[B0001-s003] False (hcx) 현재 문장은 부동산 전문가의 의견 및 예측을 담고 있으며, 구체적인 수치
[B0001-s004] False (hcx) 현재 문장은 질문 형태로 되어 있으며, 특정한 수치나 통계적인 변화를 주
[B0001-s005] False (hcx) 현재 문장은 서울과 수도권의 부동산 가격 상승에 대한 예측이므로, 이는 
[B0001-s006] False (hcx) 현재 문장은 수도권 내 특정 지역의 부동산 가격 상승 예측에 대한 의견을
[B0001-s007] True (hcx) 현재 문장은 '나머지 지역은 조금 떨어질 수도 있을 것이라고 본다.'라는
[B0001-s008] True (hcx) 현재 문장은 서울의 집값이 2~3% 정도 상승할 것이라는 구체적인 수치적
[B0001-s009] True (hcx) 경기도의 외곽 지역에서 하락 가능성을 언급하며 전체적으로 1~2% 정도의
[B0001-s010] True (hcx) 지방의 부동산 가격 변동에 대한 예측이므로 국내 집계 통계로서 KOSIS
[B0001-s011] False (hcx) 현재 문장은 수도권 주택 공급이 줄고 있다는 일반적인 상황을 언급하지만,
[B0001-s012] False (hcx) 현재 문장은 '공급 부족은 가격 상승에 가장 큰 원인이다.'라는 단순한 
[B0001-s013] False (hcx) 현재 문장은 다주택자들의 물건 출회와 관련된 정책에 대한 의견을 제시하지
[B0001-s014] False (hcx) 현재 문장은 정치적 불확실성이 부동산 가격에 미치는 영향에 대한 의견이지
[B0001-s015] Fa

In [ ]:
!python filter_true_claims.py --input is_claim_result.csv --output claims_filtered.csv

생성: claims_filtered.csv
전체 문장: 840 | is_claim=True: 206 (25%) | 그중 confidence=low: 1 | 숫자 없는 True(추세 주장): 39


In [ ]:
!python extract_hcx.py --input claims_filtered.csv --output hcx_extracted.csv

[B0001-s001] ok (3 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s007] ok (1 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s008] ok (2 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s009] ok (2 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s010] ok (2 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s020] ok (1 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s031] ok (1 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0001-s046] ok (1 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0002-s008] ok (1 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0004-s001] ok (2 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0004-s002] ok (4 행, repair=Y, fallback=1, binding=0, period_removed=1)
[B0004-s003] ok (4 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0004-s004] ok (4 행, repair=N, fallback=0, binding=0, period_removed=0)
[B0004-s005] ok (1 행, repair=N, fallback=0, binding

## 1. 스크립트 & 데이터 업로드

파일 선택 창에서 `extract_hcx.py`, `extract_article_claims_hcx.py`, 기사 원문 CSV를
한 번에 선택해서 업로드하세요.


In [ ]:
from google.colab import files

uploaded = files.upload()
print("업로드됨:", list(uploaded.keys()))


업로드됨: []


### (대안) Google Drive에 스크립트를 미리 저장해두셨다면


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil, os
# SRC_DIR = '/content/drive/MyDrive/여기에_실제_폴더_경로'
# for fname in ['extract_hcx.py', 'extract_article_claims_hcx.py']:
#     shutil.copy(os.path.join(SRC_DIR, fname), fname)
# print('스크립트 복사 완료')


## 2. CLOVA_API_KEY 로드 (Colab 보안 비밀)


In [ ]:
import os
from google.colab import userdata

key = userdata.get('CLOVA_API_KEY')
os.environ['CLOVA_API_KEY'] = key or ''
print('키 로드 완료 (길이 {}자)'.format(len(key)) if key else
      '키를 찾을 수 없습니다 - 보안 비밀 이름()과 "노트북 액세스" 토글을 확인하세요')


키 로드 완료 (길이 39자)


In [4]:
import os
from google.colab import userdata
os.environ["CLOVA_API_KEY"] = userdata.get("CLOVA_API_KEY")
os.environ["KOSIS_API_KEY"] = userdata.get("KOSIS_API_KEY")
print("CLOVA:", bool(os.environ.get("CLOVA_API_KEY")), "| KOSIS:", bool(os.environ.get("KOSIS_API_KEY")))

CLOVA: True | KOSIS: True


In [ ]:
# 1) 놓친 도메인만 같은 파일에 이어붙이기
!python crawl_kosis_catalog.py --output kosis_full_catalog.csv \
    --top-category-keywords "도소매ㆍ서비스,소득ㆍ소비ㆍ자산" --resume --max-tables 5000
!python crawl_kosis_catalog.py --output kosis_full_catalog.csv \
    --top-category-keywords "물가,인구,정부ㆍ재정,임금,보건,복지" --resume --max-tables 5000



kosis_full_catalog.csv 가 이미 있습니다 - 기존 70개 표는 건너뛰고 이어서 append 합니다.
(카테고리 트리는 처음부터 다시 훑지만, 이미 모은 표는 중복으로 다시 안 씁니다.)
크롤링 시작: vw_cd=MT_ZTITLE, max_depth=무제한, top_category_keywords=['도소매ㆍ서비스', '소득ㆍ소비ㆍ자산'], max_tables=5000
진행: 카테고리 호출 20개 완료, 지금까지 표 266개, 대기 중인 카테고리 29개
실패 (parent_id='113_11308_011', path=도소매ㆍ서비스/콘텐츠산업조사/지식정보산업): ConnectionError: HTTPSConnectionPool(host='kosis.kr', port=443): Max retries exceeded with url: /openapi/statisticsList.do?method=getList&apiKey=ZTFiMTMzZDliNDQyYmRlMWI5MjJjMmUxNzgyODg3NGM%3D&vwCd=MT_ZTITLE&parentListId=113_11308_011&format=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7d965f414770>: Failed to resolve 'kosis.kr' ([Errno -5] No address associated with hostname)"))
실패 (parent_id='113_11308_010', path=도소매ㆍ서비스/콘텐츠산업조사/캐릭터사업): ConnectionError: HTTPSConnectionPool(host='kosis.kr', port=443): Max retries exceeded with url: /openapi/statisticsList.do?method=getList&apiKey=ZTFiMTMzZDliNDQyYmRlMWI5MjJjMmUxNzgyODg3NGM%3D&

In [13]:
# 2) enrichment - 새 표만 처리됨
!python enrich_kosis_catalog_meta.py --input kosis_full_catalog.csv \
    --output kosis_metadata_summary_enriched.csv

이어받기: 34개 표는 이미 완료되어 건너뜁니다.
[20/10040] 진행: 성공 20 | 실패 0 | 건너뜀 0
[40/10040] 진행: 성공 40 | 실패 0 | 건너뜀 0
[60/10040] 진행: 성공 60 | 실패 0 | 건너뜀 0
[80/10040] 진행: 성공 80 | 실패 0 | 건너뜀 0
[100/10040] 진행: 성공 100 | 실패 0 | 건너뜀 0
[120/10040] 진행: 성공 120 | 실패 0 | 건너뜀 0
[140/10040] 진행: 성공 140 | 실패 0 | 건너뜀 0
[160/10040] 진행: 성공 160 | 실패 0 | 건너뜀 0
[180/10040] 진행: 성공 180 | 실패 0 | 건너뜀 0
[200/10040] 진행: 성공 200 | 실패 0 | 건너뜀 0
[220/10040] 진행: 성공 220 | 실패 0 | 건너뜀 0
[240/10040] 진행: 성공 240 | 실패 0 | 건너뜀 0
[260/10040] 진행: 성공 260 | 실패 0 | 건너뜀 0
[280/10040] 진행: 성공 280 | 실패 0 | 건너뜀 0
[300/10040] 진행: 성공 300 | 실패 0 | 건너뜀 0
[320/10040] 진행: 성공 320 | 실패 0 | 건너뜀 0
[340/10040] 진행: 성공 340 | 실패 0 | 건너뜀 0
[360/10040] 진행: 성공 360 | 실패 0 | 건너뜀 0
[380/10040] 진행: 성공 380 | 실패 0 | 건너뜀 0
[400/10040] 진행: 성공 400 | 실패 0 | 건너뜀 0
[420/10040] 진행: 성공 420 | 실패 0 | 건너뜀 0
[440/10040] 진행: 성공 440 | 실패 0 | 건너뜀 0
[460/10040] 진행: 성공 460 | 실패 0 | 건너뜀 0
[480/10040] 진행: 성공 480 | 실패 0 | 건너뜀 0
[500/10040] 진행: 성공 500 | 실패 0 | 건너뜀 0
[520/10040] 진행: 성공 520 | 실패 0 

In [ ]:
!python filter_catalog_by_relevance.py \
    --catalog kosis_full_catalog.csv \
    --claims kosis_ready_plus_improvable.csv \
    --output kosis_full_catalog_trimmed.csv \
    --dropped-output kosis_full_catalog_dropped.csv

claim 142건에서 뽑은 키워드 251개
생성: kosis_full_catalog_trimmed.csv
전체 10074개 표 중 유지 10074개, 제거 0개
제거된 표 목록(참고용): kosis_full_catalog_dropped.csv


In [14]:
!python enrich_kosis_catalog_meta.py \
    --input kosis_full_catalog_trimmed.csv \
    --output kosis_metadata_summary_enriched.csv

Traceback (most recent call last):
  File "/content/enrich_kosis_catalog_meta.py", line 229, in <module>
    main()
  File "/content/enrich_kosis_catalog_meta.py", line 225, in main
    run(args)
  File "/content/enrich_kosis_catalog_meta.py", line 108, in run
    tables = read_csv(args.input)
             ^^^^^^^^^^^^^^^^^^^^
  File "/content/enrich_kosis_catalog_meta.py", line 37, in read_csv
    with open(path, encoding="utf-8-sig", newline="") as file:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'kosis_full_catalog_trimmed.csv'


In [ ]:

# 3) 임베딩 - 새 표만 처리됨
!python embed_kosis_catalog_local.py --catalog kosis_metadata_summary_enriched.csv \
    --output kosis_stat_catalog_embeddings_v2.csv

[1/10030] OK 174/DT_163002_B002 (dim=1024)
[2/10030] OK 174/DT_163002_B0021 (dim=1024)
[3/10030] OK 174/DT_163002_B003 (dim=1024)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 464, in _make_request
    self._validate_conn(conn)
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 1093, in _validate_conn
    conn.connect()
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 790, in connect
    sock_and_verified = _ssl_wrap_socket_and_match_hostname(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 969, in _ssl_wrap_socket_and_match_hostname
    ssl_sock = ssl_wrap_socket(
               ^^^^^^^^^^^^^^^^
  File "/us

In [10]:
import pandas as pd
df = pd.read_csv("kosis_full_catalog_trimmed.csv")
# 겹치는 키워드가 1개뿐인 것 제외 (덜 확실한 매칭)
df["_kw_count"] = df["_matched_keywords"].fillna("").apply(lambda x: len(x.split(",")) if x else 0)
df_strict = df[df["_kw_count"] >= 2]
df_strict.to_csv("kosis_full_catalog_trimmed_strict.csv", index=False, encoding="utf-8-sig")
print(len(df), "->", len(df_strict))

FileNotFoundError: [Errno 2] No such file or directory: 'kosis_full_catalog_trimmed.csv'

In [9]:
!python enrich_kosis_catalog_meta.py \
    --input kosis_full_catalog_trimmed.csv \
    --output kosis_metadata_summary_enriched.csv \
    --limit 300

Traceback (most recent call last):
  File "/content/enrich_kosis_catalog_meta.py", line 229, in <module>
    main()
  File "/content/enrich_kosis_catalog_meta.py", line 225, in main
    run(args)
  File "/content/enrich_kosis_catalog_meta.py", line 108, in run
    tables = read_csv(args.input)
             ^^^^^^^^^^^^^^^^^^^^
  File "/content/enrich_kosis_catalog_meta.py", line 37, in read_csv
    with open(path, encoding="utf-8-sig", newline="") as file:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: 'kosis_full_catalog_trimmed.csv'


In [ ]:
!python filter_catalog_by_relevance.py \
    --catalog kosis_full_catalog.csv \
    --claims kosis_ready_plus_improvable.csv \
    --output kosis_full_catalog_trimmed.csv \
    --dropped-output kosis_full_catalog_dropped.csv

claim 142건에서 뽑은 키워드 251개
생성: kosis_full_catalog_trimmed.csv
전체 10074개 표 중 유지 10074개, 제거 0개
제거된 표 목록(참고용): kosis_full_catalog_dropped.csv


In [ ]:
# 1) 필터 다시 (이번엔 실제로 걸러질 겁니다)
!python filter_catalog_by_relevance.py \
    --catalog kosis_full_catalog.csv \
    --claims kosis_ready_plus_improvable.csv \
    --output kosis_full_catalog_trimmed.csv \
    --dropped-output kosis_full_catalog_dropped.csv

claim 142건에서 뽑은 키워드 229개
생성: kosis_full_catalog_trimmed.csv
전체 10074개 표 중 유지 5282개, 제거 4792개
제거된 표 목록(참고용): kosis_full_catalog_dropped.csv


In [ ]:


# 4) meta-index - 새 표만 처리됨 (같은 kosis_meta_index_v2.csv에 이어붙임)
!python build_kosis_meta_index.py --catalog kosis_metadata_summary_enriched.csv \
    --output kosis_meta_index_v2.csv

이어받기: kosis_meta_index_v2.csv 에 이미 있는 2075개 표는 건너뜁니다 (카탈로그에 새로 추가된 표만 처리).
[1/10065] OK 174/DT_163002_B002 직종별 현원 총괄 (25행)
[2/10065] OK 174/DT_163002_B0021 국가공무원 직종별 현원 (22행)
[3/10065] OK 174/DT_163002_B003 직종별 현원 : 행정·기술·관리운영직 (16행)
[4/10065] OK 174/DT_163002_B0031 직종별 현원 : 일반계약직 (14행)
[5/10065] OK 174/DT_163002_B004 직종별 현원 : 우정직 (14행)
Traceback (most recent call last):
  File "/content/build_kosis_meta_index.py", line 143, in <module>
    main()
  File "/content/build_kosis_meta_index.py", line 139, in main
    run(args)
  File "/content/build_kosis_meta_index.py", line 87, in run
    meta_rows = get_meta(org_id, tbl_id, meta_type="ITM")
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/kosis_api_test.py", line 111, in get_meta
    res = requests.get(META_URL, params=params, timeout=10)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/requests/api.py", line 73, in get
    return request("get", url, pa

In [ ]:

df_enriched = pd.read_csv("kosis_metadata_summary_enriched.csv")
df_trimmed = pd.read_csv("kosis_full_catalog_trimmed.csv")  # 관련 있는 것만 걸러진 목록

keep_keys = set(zip(df_trimmed["org_id"].astype(str), df_trimmed["tbl_id"].astype(str)))
df_enriched_filtered = df_enriched[
    df_enriched.apply(lambda r: (str(r["ORG_ID"]), str(r["TBL_ID"])) in keep_keys, axis=1)
]
df_enriched_filtered.to_csv("kosis_metadata_summary_enriched_filtered.csv", index=False, encoding="utf-8-sig")
print(len(df_enriched), "->", len(df_enriched_filtered))

10065 -> 5282


In [ ]:
df_meta = pd.read_csv("kosis_meta_index_v2.csv")
df_meta_filtered = df_meta[
    df_meta.apply(lambda r: (str(r["org_id"]), str(r["tbl_id"])) in keep_keys, axis=1)
]
df_meta_filtered.to_csv("kosis_meta_index_v2_filtered.csv", index=False, encoding="utf-8-sig")
print(len(df_meta), "->", len(df_meta_filtered))

145139 -> 0


In [ ]:
df_trimmed = pd.read_csv("kosis_full_catalog_trimmed.csv")
df_meta = pd.read_csv("kosis_meta_index_v2.csv")

print("trimmed 컬럼:", df_trimmed.columns.tolist())
print("meta 컬럼:", df_meta.columns.tolist())
print()
print("trimmed org_id/tbl_id 샘플:")
print(df_trimmed[["org_id","tbl_id"]].head(3))
print(df_trimmed.dtypes[["org_id","tbl_id"]])
print()
print("meta org_id/tbl_id 샘플:")
print(df_meta[["org_id","tbl_id"]].head(3))
print(df_meta.dtypes[["org_id","tbl_id"]])

trimmed 컬럼: ['org_id', 'tbl_id', 'tbl_nm', 'category_path', 'depth', '_matched_keywords']
meta 컬럼: ['OBJ_ID', 'TBL_ID', 'UNIT_ID', 'ORG_ID', 'OBJ_NM', 'ITM_NM', 'UNIT_ENG_NM', 'ITM_ID', 'UNIT_NM', 'OBJ_NM_ENG', 'org_id', 'tbl_id', 'OBJ_ID_SN', 'ITM_NM_ENG', 'UP_ITM_ID']

trimmed org_id/tbl_id 샘플:
   org_id     tbl_id
0     101  DT_FR0001
1     101  DT_FR0002
2     101  DT_FR0003
org_id     int64
tbl_id    object
dtype: object

meta org_id/tbl_id 샘플:
   org_id           tbl_id
0     102  DT_AS10203-N001
1     102  DT_AS10203-N001
2     102  DT_AS10203-N001
org_id     int64
tbl_id    object
dtype: object


## 3. 입력 파일 지정

업로드한 기사 원문 CSV 파일명으로 아래 변수를 수정하세요.


In [ ]:
ARTICLES_CSV = "articles.csv"  # 업로드한 실제 파일명으로 수정


## 4. 실행 (HCX 호출)

- `LIMIT`: 앞에서부터 N개 기사만 처리. `0`이면 전체 처리.
- `MAX_CHARS`: 기사 본문이 이보다 길면 문단 단위로 나눠서(chunk) 여러 번 호출합니다.
  chunk로 나뉘면 "기사 전체 맥락" 효과가 chunk 범위로 줄어드니, 웬만하면 기사 하나가
  통째로 들어갈 수 있는 값으로 넉넉히 두는 걸 권장합니다.
- 먼저 `LIMIT`을 작게 두고 결과를 확인한 다음 `0`으로 바꿔 전체 처리하세요.


In [ ]:
LIMIT = 0          # 0 = 전체 처리
MAX_CHARS = 3000   # 기사 본문 chunk 최대 길이

!python extract_article_claims_hcx.py \
    --input {ARTICLES_CSV} \
    --output hcx_article_extracted.csv \
    --limit {LIMIT} \
    --max-chars {MAX_CHARS}


이어받기: 0개 기사 완료됨
[A0001] ok (claim 0건, 행 0개, chunk 1개, grounding 실패 0건)
[A0002] ok (claim 0건, 행 0개, chunk 1개, grounding 실패 0건)
[A0003] ok (claim 1건, 행 3개, chunk 1개, grounding 실패 1건)
[A0004] ok (claim 4건, 행 9개, chunk 1개, grounding 실패 0건)
[A0005] ok (claim 1건, 행 1개, chunk 1개, grounding 실패 0건)
[A0006] 실패: JSONDecodeError: Expecting ',' delimiter: line 402 column 6 (char 12277)
[A0007] ok (claim 5건, 행 7개, chunk 1개, grounding 실패 1건)
[A0008] ok (claim 2건, 행 2개, chunk 1개, grounding 실패 0건)
[A0009] 실패: JSONDecodeError: Expecting ',' delimiter: line 353 column 6 (char 11612)
[A0010] 실패: JSONDecodeError: Expecting ',' delimiter: line 398 column 6 (char 12349)
[A0011] ok (claim 1건, 행 1개, chunk 1개, grounding 실패 0건)
[A0012] ok (claim 1건, 행 2개, chunk 1개, grounding 실패 0건)
[A0013] ok (claim 2건, 행 2개, chunk 1개, grounding 실패 0건)
[A0014] ok (claim 0건, 행 0개, chunk 1개, grounding 실패 0건)
[A0015] ok (claim 1건, 행 2개, chunk 1개, grounding 실패 0건)
[A0016] ok (claim 0건, 행 0개, chunk 1개, grounding 실패 0건)
[A0017] ok (cl

In [ ]:
import os
print(os.listdir())

['.config', '.ipynb_checkpoints', '뉴스_데이터_규칙기반정제Top50.csv', 'extract_article_claims_hcx.py', 'extract_hcx .py', 'sample_data']


## 5. 결과 확인


In [ ]:
import pandas as pd

df = pd.read_csv("hcx_article_extracted.csv")
print(len(df), "행 (claim당 measurement 여러 행일 수 있음)")
print()
print("기사 수:", df["article_id"].nunique(), "| claim 수:", df["claim_id"].nunique())
print("grounding 실패(claim_grounded=N):", (df["claim_grounded"] == "N").sum(), "건")
print("규칙 fallback 보정된 행:", (df["measurement_source"] == "rule_fallback").sum(), "건")
print("needs_review=Y:", (df["needs_review"] == "Y").sum(), "건")
df.head(10)


138 행 (claim당 measurement 여러 행일 수 있음)

기사 수: 26 | claim 수: 73
grounding 실패(claim_grounded=N): 32 건
규칙 fallback 보정된 행: 41 건
needs_review=Y: 105 건


,claim_id,claim_measurement_id,article_id,title,date,url,claim_text,prev_sentence,next_sentence,claim_domain_scope,...,review_reason,measurement_repaired,measurement_fallback_count,measurement_binding_fallback_count,extraction_model,prompt_version,extracted_at,claim_grounded,extraction_scope,chunk_index
0,A0003-c001,A0003-c001-m1,A0003,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원,2025-01-01,https://www.chosun.com/economy/economy_general...,최저임금이 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7...,-,-,국내공식통계,...,measurement_rule_fallback:1;measurement_bindin...,Y,1,1,HCX-007,article-combined-v1.0,2026-07-30,N,article_combined,0
1,A0003-c001,A0003-c001-m2,A0003,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원,2025-01-01,https://www.chosun.com/economy/economy_general...,최저임금이 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7...,-,-,국내공식통계,...,measurement_rule_fallback:1;measurement_bindin...,Y,1,1,HCX-007,article-combined-v1.0,2026-07-30,N,article_combined,0
2,A0003-c001,A0003-c001-m3,A0003,최저임금 1만30원으로 인상… 육아휴직 급여 최대 월 250만원,2025-01-01,https://www.chosun.com/economy/economy_general...,최저임금이 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7...,-,-,국내공식통계,...,measurement_rule_fallback:1;measurement_bindin...,Y,1,1,HCX-007,article-combined-v1.0,2026-07-30,N,article_combined,0
3,A0004-c001,A0004-c001-m1,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,국제로봇연맹(IFR)에 따르면 2023년 우리나라는 직원 1만명당 로봇 1012대를...,-,-,국내공식통계,...,measurement_rule_fallback:1,Y,1,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0
4,A0004-c001,A0004-c001-m2,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,국제로봇연맹(IFR)에 따르면 2023년 우리나라는 직원 1만명당 로봇 1012대를...,-,-,국내공식통계,...,measurement_rule_fallback:1,Y,1,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0
5,A0004-c002,A0004-c002-m1,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,전 세계 평균(1만명당 162대)의 6배가 넘는 수치다.,-,-,국내공식통계,...,measurement_rule_fallback:2,Y,2,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0
6,A0004-c002,A0004-c002-m2,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,전 세계 평균(1만명당 162대)의 6배가 넘는 수치다.,-,-,국내공식통계,...,measurement_rule_fallback:2,Y,2,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0
7,A0004-c002,A0004-c002-m3,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,전 세계 평균(1만명당 162대)의 6배가 넘는 수치다.,-,-,국내공식통계,...,measurement_rule_fallback:2,Y,2,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0
8,A0004-c003,A0004-c003-m1,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,전 세계 평균(1만명당 162대)의 6배가 넘는 수치다.,-,-,국내공식통계,...,measurement_rule_fallback:1,Y,1,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0
9,A0004-c003,A0004-c003-m2,A0004,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,https://www.chosun.com/economy/economy_general...,전 세계 평균(1만명당 162대)의 6배가 넘는 수치다.,-,-,국내공식통계,...,measurement_rule_fallback:1,Y,1,0,HCX-007,article-combined-v1.0,2026-07-30,Y,article_combined,0


### 검토가 필요한 행만 모아보기

`claim_grounded=N`(기사에 없는 문장을 지어낸 경우)이나 `needs_review=Y`인 행은
우선적으로 사람이 확인하는 걸 권장합니다.


In [ ]:
review_cols = ["claim_id", "claim_text", "value", "unit", "measurement_indicator",
               "measurement_period", "claim_grounded", "needs_review", "review_reason"]

df_review = df[(df["claim_grounded"] == "N") | (df["needs_review"] == "Y")]
print(len(df_review), "건 검토 필요")
df_review[review_cols]


106 건 검토 필요


,claim_id,claim_text,value,unit,measurement_indicator,measurement_period,claim_grounded,needs_review,review_reason
0,A0003-c001,최저임금이 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7...,10030,원,최저임금,2025,N,Y,measurement_rule_fallback:1;measurement_bindin...
1,A0003-c001,최저임금이 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7...,9860,원,최저임금,2025,N,Y,measurement_rule_fallback:1;measurement_bindin...
2,A0003-c001,최저임금이 시간당 1만30원 =최저임금이 시간당 9860원에서 1만30원으로 1.7...,1.7,%,최저임금,2025,N,Y,measurement_rule_fallback:1;measurement_bindin...
3,A0004-c001,국제로봇연맹(IFR)에 따르면 2023년 우리나라는 직원 1만명당 로봇 1012대를...,1012,대,로봇 밀도,2023,Y,Y,measurement_rule_fallback:1
4,A0004-c001,국제로봇연맹(IFR)에 따르면 2023년 우리나라는 직원 1만명당 로봇 1012대를...,10000,명,로봇 밀도,2023,Y,Y,measurement_rule_fallback:1
...,...,...,...,...,...,...,...,...,...
129,A0049-c002,현대건설의 첫 수주 이후 59년 만의 성과다.,59,년,경과 시간,2025,Y,Y,measurement_binding_fallback:1;measurement_per...
134,A0049-c007,오는 3월 최종 계약을 앞둔 24조원 규모 체코 두코바니 원전은 대우건설이 시공할 ...,24000000000000,원,체코 두코바니 원전 수주 예상액,-,Y,Y,measurement_period_ungrounded:1
135,A0050-c001,통계청 자료에 따르면 국내 부업자 수는 2023년 1분기 45만명에서 2024년 1...,550000,명,부업자 수,2024,Y,Y,measurement_rule_fallback:1
136,A0050-c001,통계청 자료에 따르면 국내 부업자 수는 2023년 1분기 45만명에서 2024년 1...,450000,명,부업자 수,2023,Y,Y,measurement_rule_fallback:1


## (선택, 고급) 이전 실행 결과와 회귀 비교

프롬프트나 모델을 바꿔가며 재실행할 때, 이전 결과에서 값(value)이 비어 있던
claim들이 이번엔 채워졌는지 확인하고 싶다면 `measurement_regression.py`(문장 단위
파이프라인에서 쓰던 스크립트)를 그대로 재사용할 수 있습니다. `--baseline`에는
**이전 실행 결과 CSV**를 넣으세요. 처음 실행이라 비교 대상이 없다면 건너뛰어도 됩니다.


In [ ]:
# measurement_regression.py 도 업로드했다면 아래 주석을 풀어 사용하세요.
# PREVIOUS_EXTRACTED = "이전_hcx_article_extracted.csv"  # 이전 실행 결과 파일명
#
# !python measurement_regression.py audit \
#     --baseline {PREVIOUS_EXTRACTED} \
#     --candidate hcx_article_extracted.csv \
#     --report audit_report.csv
#
# import json
# df_report = pd.read_csv("audit_report.csv")
# print(df_report["status"].value_counts())
# json.loads(open("audit_report.summary.json", encoding="utf-8").read())


## 6. 결과 다운로드


In [ ]:
from google.colab import files

files.download("hcx_article_extracted.csv")


In [ ]:
from google.colab import files

uploaded = files.upload()  # prepare_kosis_mapping_input.py 선택
print("업로드됨:", list(uploaded.keys()))

Saving prepare_kosis_mapping_input.py to prepare_kosis_mapping_input.py
업로드됨: ['prepare_kosis_mapping_input.py']


In [ ]:
!python prepare_kosis_mapping_input.py \
    --input hcx_extracted.csv \
    --output kosis_ready.csv \
    --rejected-output kosis_rejected.csv


input=421 ready=114 rejected=307
rejection_counts=OUT_OF_KOSIS_SCOPE:136, NOT_KOSIS_VALUE:64, NO_MEASUREMENT:53, PERIOD_MISSING:27, ROLE_NOT_DIRECT_TARGET:16, RANK_NOT_DIRECTLY_COMPARABLE:5, UNIT_UNSUPPORTED:4, BINDING_NOT_CONFIRMED:1, VALUE_MISSING:1
saved=kosis_ready.csv
rejected=kosis_rejected.csv


In [ ]:
import pandas as pd

df_ready = pd.read_csv("kosis_ready.csv")
df_rejected = pd.read_csv("kosis_rejected.csv")

print(f"ready: {len(df_ready)}건 | rejected: {len(df_rejected)}건 "
      f"(전체 {len(df_ready) + len(df_rejected)}건 중 {len(df_ready)/(len(df_ready)+len(df_rejected)):.1%})")
print()
print("배제 사유 분포:")
print(df_rejected["mapping_exclusion_code"].value_counts())

ready: 114건 | rejected: 307건 (전체 421건 중 27.1%)

배제 사유 분포:
mapping_exclusion_code
OUT_OF_KOSIS_SCOPE              136
NOT_KOSIS_VALUE                  64
NO_MEASUREMENT                   53
PERIOD_MISSING                   27
ROLE_NOT_DIRECT_TARGET           16
RANK_NOT_DIRECTLY_COMPARABLE      5
UNIT_UNSUPPORTED                  4
BINDING_NOT_CONFIRMED             1
VALUE_MISSING                     1
Name: count, dtype: int64


In [ ]:
# ready 로 통과한 값 미리보기
cols = ["claim_id", "claim_text", "indicator", "value", "unit", "canonical_unit",
        "semantic_type", "period", "prd_se", "entity_type"]
df_ready[cols]

,claim_id,claim_text,indicator,value,unit,canonical_unit,semantic_type,period,prd_se,entity_type
0,B0006-s002,"지난 1일까지 설선물 매출을 지난 설 동기간과 비교했을 때 롯데백화점은 45%, 신...",설선물 매출 증가율,4.500000e+01,%,%,rate_change,202501,M,item
1,B0006-s002,"지난 1일까지 설선물 매출을 지난 설 동기간과 비교했을 때 롯데백화점은 45%, 신...",설선물 매출 증가율,7.240000e+01,%,%,rate_change,202501,M,item
2,B0006-s002,"지난 1일까지 설선물 매출을 지난 설 동기간과 비교했을 때 롯데백화점은 45%, 신...",설선물 매출 증가율,7.180000e+01,%,%,rate_change,202501,M,item
3,B0006-s002,"지난 1일까지 설선물 매출을 지난 설 동기간과 비교했을 때 롯데백화점은 45%, 신...",설선물 매출 증가율,3.200000e+01,%,%,rate_change,202501,M,item
4,B0006-s011,"롯데백화점은 5일까지 롯데백화점 전점에서 축산, 수산, 청과, 그로서리 등에서 약 ...",설 사전 예약 판매 품목 수,2.300000e+02,개,개,count,202501,M,unspecified
...,...,...,...,...,...,...,...,...,...,...
109,B0046-s005,직전 최저치는 2023년 4분기의 29조8000억원이다.,순자금 운용액,2.980000e+13,원,원,amount,2023,Y,unspecified
110,B0046-s007,통계청에 따르면 작년 3분기 가계소득은 전 분기보다 5.9% 늘었다.,가계소득,5.900000e+00,%,%,rate_change,2024Q3,Q,unspecified
111,B0046-s009,가계 및 비영리단체의 자금 조달액 중 주택 담보대출금도 작년 3분기에 전 분기보다 ...,가계 및 비영리단체의 자금 조달액,1.940000e+13,원,원,amount,2024Q3,Q,item
112,B0047-s009,"종신보험은 보험료 납입 기간이 20~30년인데, 현재 보험료 납입이 끝난 계약은 약...",보험료 납입 완료 계약 건수,3.620000e+06,건,건,count,2025,Y,unspecified


In [ ]:
# BINDING_NOT_CONFIRMED / PERIODICITY_MISSING 처럼 "개선하면 ready 될 수 있는" 케이스만 모아보기
IMPROVABLE_CODES = {"BINDING_NOT_CONFIRMED", "PERIODICITY_MISSING", "PERIOD_MISSING"}
review_cols = ["claim_id", "claim_text", "value", "unit", "measurement_indicator",
               "measurement_period", "measurement_prd_se", "measurement_binding_source",
               "mapping_exclusion_code", "mapping_exclusion_reason"]

df_improvable = df_rejected[df_rejected["mapping_exclusion_code"].isin(IMPROVABLE_CODES)]
print(len(df_improvable), "건 - 프롬프트/보정 로직 개선 시 ready 로 넘어올 여지 있음")
df_improvable[review_cols]


28 건 - 프롬프트/보정 로직 개선 시 ready 로 넘어올 여지 있음


,claim_id,claim_text,value,unit,measurement_indicator,measurement_period,measurement_prd_se,measurement_binding_source,mapping_exclusion_code,mapping_exclusion_reason
54,B0011-s000,0 작년 4분기 실적 발표 시즌이 다가온 가운데 상장사들의 성적표가 3개월 전보다 ...,16.5,%,영업이익 변화율,-,-,hcx,PERIOD_MISSING,measurement period 없음
60,B0011-s006,3일 기준 증권사 3개 이상이 추정치를 제시한 코스피 상장사 110곳의 총 영업이익...,39653100000000,원,영업이익,-,-,hcx,PERIOD_MISSING,measurement period 없음
63,B0011-s007,"2023년 4분기 영업이익 합 19조7942억원과 비교하면 2배 수준이지만, 비교군...",87,개,영업이익,2024Q4,Q,rule_fallback,BINDING_NOT_CONFIRMED,measurement_binding_source=rule_fallback
71,B0016-s015,지난달 29일 사고 이후 8일간 공항에서 활동한 자원봉사자는 총 5509명으로 집계됐다.,5509,명,자원봉사자 수,-,-,hcx,PERIOD_MISSING,measurement period 없음
77,B0016-s022,"제주도관광협회에 따르면, 제주항공 참사 이튿날인 지난달 30일부터 지난 5일까지 일...",208516,명,관광객 수,-,-,hcx,PERIOD_MISSING,measurement period 없음
78,B0016-s022,"제주도관광협회에 따르면, 제주항공 참사 이튿날인 지난달 30일부터 지난 5일까지 일...",31495,명,감소한 관광객 수,-,-,hcx,PERIOD_MISSING,measurement period 없음
79,B0016-s022,"제주도관광협회에 따르면, 제주항공 참사 이튿날인 지난달 30일부터 지난 5일까지 일...",13.1,%,관광객 감소율,-,-,hcx,PERIOD_MISSING,measurement period 없음
80,B0016-s023,이 가운데 내국인 관광객은 전년 대비 3만7105명(16.7%) 줄었다.,37105,명,내국인 관광객 수,-,-,hcx,PERIOD_MISSING,measurement period 없음
81,B0016-s023,이 가운데 내국인 관광객은 전년 대비 3만7105명(16.7%) 줄었다.,16.7,%,내국인 관광객 증감률,-,-,hcx,PERIOD_MISSING,measurement period 없음
95,B0019-s001,"5일 한국농수산식품유통공사(aT)에 따르면, 지난 3일 기준 배추의 평균 소매가격은...",5027,원,배추 소매가격,-,-,hcx,PERIOD_MISSING,measurement period 없음


In [ ]:
import pandas as pd

df_ready = pd.read_csv("kosis_ready.csv")
df_rejected = pd.read_csv("kosis_rejected.csv")

IMPROVABLE_CODES = {"BINDING_NOT_CONFIRMED", "PERIODICITY_MISSING", "PERIOD_MISSING"}
df_improvable = df_rejected[df_rejected["mapping_exclusion_code"].isin(IMPROVABLE_CODES)]

df_merged = pd.concat([df_ready, df_improvable], ignore_index=True)
df_merged.to_csv("kosis_ready_plus_improvable.csv", index=False, encoding="utf-8-sig")

print(f"ready {len(df_ready)}건 + improvable {len(df_improvable)}건 = {len(df_merged)}건")

ready 114건 + improvable 28건 = 142건


In [ ]:

df_enriched = pd.read_csv("kosis_metadata_summary_enriched.csv")
print("전체 표 개수:", len(df_enriched))

df_enriched["top_category"] = df_enriched["category_path"].str.split(" > ").str[0]
print(df_enriched["top_category"].value_counts())

전체 표 개수: 10065
top_category
정부ㆍ재정       5015
도소매ㆍ서비스     4762
소득ㆍ소비ㆍ자산     237
물가            51
Name: count, dtype: int64


In [ ]:

# 예전 배치(무역+금융, 2075개)와 이번 배치(재정+도소매+소득소비+물가, 10065개)를 합침
df_old = pd.read_csv("kosis_metadata_summary_enriched_old.csv")   # 예전 2075개짜리 파일명 확인 필요
df_new = pd.read_csv("kosis_metadata_summary_enriched.csv")       # 지금 10065개
df_combined = pd.concat([df_old, df_new], ignore_index=True).drop_duplicates(subset=["ORG_ID","TBL_ID"])
df_combined.to_csv("kosis_metadata_summary_enriched_combined.csv", index=False, encoding="utf-8-sig")
print(len(df_old), "+", len(df_new), "->", len(df_combined))

# meta-index도 똑같이
df_meta_old = pd.read_csv("kosis_meta_index_old.csv")   # 예전 무역+금융 meta-index 파일명 확인 필요
df_meta_new = pd.read_csv("kosis_meta_index_v2.csv")
df_meta_combined = pd.concat([df_meta_old, df_meta_new], ignore_index=True).drop_duplicates()
df_meta_combined.to_csv("kosis_meta_index_combined.csv", index=False, encoding="utf-8-sig")
print(len(df_meta_old), "+", len(df_meta_new), "->", len(df_meta_combined))

FileNotFoundError: [Errno 2] No such file or directory: 'kosis_metadata_summary_enriched_old.csv'

In [6]:
# 1) 무역+금융만 새로 크롤링 (완전히 새 파일명)
!python crawl_kosis_catalog.py --output kosis_full_catalog_tf.csv \
    --top-category-keywords "무역ㆍ국제수지,금융" --max-tables 3000

# 2) enrichment - 새 파일명으로 (kosis_meta_index_v2.csv는 절대 건드리지 않음)
!python enrich_kosis_catalog_meta.py \
    --input kosis_full_catalog_tf.csv \
    --output kosis_metadata_summary_enriched_tf.csv \
    --meta-index-output kosis_meta_index_tf.csv

크롤링 시작: vw_cd=MT_ZTITLE, max_depth=무제한, top_category_keywords=['무역ㆍ국제수지', '금융'], max_tables=3000
진행: 카테고리 호출 20개 완료, 지금까지 표 102개, 대기 중인 카테고리 15개
진행: 카테고리 호출 40개 완료, 지금까지 표 281개, 대기 중인 카테고리 6개
진행: 카테고리 호출 60개 완료, 지금까지 표 336개, 대기 중인 카테고리 14개
진행: 카테고리 호출 80개 완료, 지금까지 표 380개, 대기 중인 카테고리 18개
진행: 카테고리 호출 100개 완료, 지금까지 표 416개, 대기 중인 카테고리 22개
진행: 카테고리 호출 120개 완료, 지금까지 표 447개, 대기 중인 카테고리 15개
진행: 카테고리 호출 140개 완료, 지금까지 표 499개, 대기 중인 카테고리 15개
진행: 카테고리 호출 160개 완료, 지금까지 표 918개, 대기 중인 카테고리 19개
진행: 카테고리 호출 180개 완료, 지금까지 표 1105개, 대기 중인 카테고리 24개
진행: 카테고리 호출 200개 완료, 지금까지 표 1250개, 대기 중인 카테고리 18개
진행: 카테고리 호출 220개 완료, 지금까지 표 1344개, 대기 중인 카테고리 10개
진행: 카테고리 호출 240개 완료, 지금까지 표 1401개, 대기 중인 카테고리 21개
진행: 카테고리 호출 260개 완료, 지금까지 표 1541개, 대기 중인 카테고리 14개
진행: 카테고리 호출 280개 완료, 지금까지 표 1734개, 대기 중인 카테고리 10개
진행: 카테고리 호출 300개 완료, 지금까지 표 1878개, 대기 중인 카테고리 5개
진행: 카테고리 호출 320개 완료, 지금까지 표 2193개, 대기 중인 카테고리 18개
진행: 카테고리 호출 340개 완료, 지금까지 표 2616개, 대기 중인 카테고리 10개
진행: 카테고리 호출 360개 완료, 지금까지 표 2694개, 대기 중인 카테고리 6개

완료: 표 2755개 발견, 카

In [15]:
import pandas as pd

df_meta_combined = pd.concat([
    pd.read_csv("kosis_meta_index_v2.csv"),      # 재정+도소매+소득소비+물가 (10065개)
    pd.read_csv("kosis_meta_index_tf.csv"),      # 무역+금융 (새로)
], ignore_index=True).drop_duplicates()
df_meta_combined.to_csv("kosis_meta_index_final.csv", index=False, encoding="utf-8-sig")

df_summary_combined = pd.concat([
    pd.read_csv("kosis_metadata_summary_enriched.csv"),
    pd.read_csv("kosis_metadata_summary_enriched_tf.csv"),
], ignore_index=True).drop_duplicates(subset=["ORG_ID","TBL_ID"])
df_summary_combined.to_csv("kosis_metadata_summary_enriched_final.csv", index=False, encoding="utf-8-sig")

print("meta:", len(df_meta_combined), "| summary:", len(df_summary_combined))

meta: 179746 | summary: 4811


In [17]:
!python embed_kosis_catalog_local.py \
    --catalog kosis_metadata_summary_enriched_final.csv \
    --output kosis_stat_catalog_embeddings_final.csv

[1/4811] OK 174/DT_163002_B001 (dim=1024)
[2/4811] OK 174/DT_163002_B002 (dim=1024)
[3/4811] OK 174/DT_163002_B0021 (dim=1024)
[4/4811] OK 174/DT_163002_B003 (dim=1024)
[5/4811] OK 174/DT_163002_B0031 (dim=1024)
[6/4811] OK 174/DT_163002_B004 (dim=1024)
[7/4811] OK 174/DT_163002_B005 (dim=1024)
[8/4811] OK 174/DT_163002_B006 (dim=1024)
[9/4811] OK 174/DT_163002_B007 (dim=1024)
[10/4811] OK 174/DT_163002_B0071 (dim=1024)
[11/4811] OK 174/DT_163002_B008 (dim=1024)
[12/4811] OK 174/DT_163002_B0081 (dim=1024)
[13/4811] OK 174/DT_163002_B009 (dim=1024)
[14/4811] OK 174/DT_163002_B010 (dim=1024)
[15/4811] OK 174/DT_163002_B011 (dim=1024)
[16/4811] OK 174/DT_163002_B012 (dim=1024)
[17/4811] OK 174/DT_163002_B013 (dim=1024)
[18/4811] OK 174/DT_163002_B014 (dim=1024)
[19/4811] OK 174/DT_163002_B015 (dim=1024)
[20/4811] OK 174/DT_163002_B016 (dim=1024)
[21/4811] OK 174/DT_163002_B017 (dim=1024)
[22/4811] OK 174/DT_163002_B018 (dim=1024)
[23/4811] OK 174/DT_163002_B019 (dim=1024)
[24/4811] OK 174

In [22]:
!python search_kosis_catalog_local.py \
    --claims kosis_ready_plus_improvable.csv \
    --catalog-embeddings kosis_stat_catalog_embeddings_final.csv \
    --output kosis_table_candidates_final.csv \
    --top-k 3

카탈로그 4811개 테이블 로드 완료

생성: kosis_table_candidates_final.csv
claim 수: 142 | 1순위 후보 READY: 0 | 1순위 후보 NEEDS_CONFIRMATION: 142 | 후보 테이블 없음: 0


In [23]:
import pandas as pd
df_candidates = pd.read_csv("kosis_table_candidates_final.csv")
rank1 = df_candidates[df_candidates["candidate_rank"] == 1].copy()
rank1["cosine_distance"] = rank1["match_reason"].str.extract(r"cosine_distance=([\d.]+)").astype(float)

cols = ["metric_domain", "indicator", "tbl_nm", "cosine_distance"]
for domain in rank1["metric_domain"].unique():
    print(f"=== {domain} ===")
    print(rank1[rank1["metric_domain"] == domain][cols].head(3).to_string(index=False))

=== 소매·소비 ===
metric_domain  indicator         tbl_nm  cosine_distance
        소매·소비 설선물 매출 증가율 백화점 매출 동향(품목별)           0.3861
        소매·소비 설선물 매출 증가율 백화점 매출 동향(품목별)           0.3967
        소매·소비 설선물 매출 증가율 백화점 매출 동향(품목별)           0.3710
=== 재정·조세 ===
metric_domain indicator         tbl_nm  cosine_distance
        재정·조세     경제성장률    경기전망지수(가동율)           0.3752
        재정·조세  가계 여유 자금     가계대출 잔액 비중           0.3586
        재정·조세 순자금 운용 규모 금융자산 운용 비중(평균)           0.3855
=== 금융·금리 ===
metric_domain  indicator       tbl_nm  cosine_distance
        금융·금리 국채 10년물 금리 10년국채선물 거래실적           0.3872
        금융·금리       영업이익      영업이익 현황           0.3533
        금융·금리 영업이익 증가 배수 코스닥 상장사 수익현황           0.3965
=== 무역 ===
metric_domain  indicator             tbl_nm  cosine_distance
           무역  수입차 판매 대수       대차거래 현황 ; 채권           0.4149
           무역  수입차 판매 대수       대차거래 현황 ; 채권           0.3575
           무역 수입차 판매 증감률 음악산업 : 수출 및 수입액 현황           0.3857
=== 소득·임금 ===
metric_domain ind

In [27]:
import pandas as pd
df = pd.read_csv("kosis_metadata_summary_enriched_final.csv")
price_tables = df[df["category_path"].str.contains("물가", na=False)]
print(len(price_tables), "개 물가 표")
print(price_tables["TBL_NM"].tolist())

0 개 물가 표
[]


In [29]:
import pandas as pd
import os

for f in ["kosis_full_catalog_tf.csv", "kosis_metadata_summary_enriched_tf.csv", "kosis_meta_index_tf.csv"]:
    if os.path.exists(f):
        print(f, "->", len(pd.read_csv(f)), "행")
    else:
        print(f, "-> 파일 없음")

kosis_full_catalog_tf.csv -> 2755 행
kosis_metadata_summary_enriched_tf.csv -> 2753 행
kosis_meta_index_tf.csv -> 179746 행


In [25]:
!python kosis_validate_mapping_candidates.py \
    --input kosis_table_candidates_final.csv \
    --meta-index kosis_meta_index_final.csv \
    --output kosis_validated_final.csv \
    --max-combinations 20 --limit 0

candidate_rows=426 max_combinations_per_row=20 estimated_api_calls<=8520
validated_rows=426 output=kosis_validated_final.csv


In [26]:
df_validated = pd.read_csv("kosis_validated_final.csv")
print(df_validated.groupby("metric_domain")["mapping_status"].value_counts())

metric_domain  mapping_status    
금융·금리          MAPPING_FAILED         39
               NOT_EVALUATED          20
               NEEDS_CONFIRMATION      1
무역             MAPPING_FAILED         83
               NOT_EVALUATED          48
               NEEDS_CONFIRMATION     13
물가             MAPPING_FAILED         10
               NOT_EVALUATED           6
               NEEDS_CONFIRMATION      2
보건·복지          MAPPING_FAILED          2
               NOT_EVALUATED           1
소득·임금          MAPPING_FAILED          2
               NOT_EVALUATED           1
소매·소비          MAPPING_FAILED        108
               NOT_EVALUATED          55
               NEEDS_CONFIRMATION      2
인구             MAPPING_FAILED          8
               NOT_EVALUATED           4
재정·조세          MAPPING_FAILED         12
               NOT_EVALUATED           7
               NEEDS_CONFIRMATION      2
Name: count, dtype: int64


In [30]:
!python enrich_kosis_catalog_meta.py \
      --input kosis_full_catalog_tf.csv \
      --output kosis_metadata_summary_enriched_tf.csv \
      --meta-index-output kosis_meta_index_tf.csv

이어받기: 2753개 표는 이미 완료되어 건너뜁니다.
[1/2] 실패 301/DT_105Y001: JSONDecodeError: Invalid \escape: line 1 column 3211 (char 3210)

완료: 성공 1 | 실패 1 | 건너뜀(이미 완료) 0
저장: kosis_metadata_summary_enriched_tf.csv
meta-index 동시 저장: kosis_meta_index_tf.csv (build_kosis_meta_index.py 별도 실행 불필요)


In [31]:
!python crawl_kosis_catalog.py --output kosis_full_catalog_tf.csv \
      --top-category-keywords "무역ㆍ국제수지,금융" --max-tables 300

크롤링 시작: vw_cd=MT_ZTITLE, max_depth=무제한, top_category_keywords=['무역ㆍ국제수지', '금융'], max_tables=300
진행: 카테고리 호출 20개 완료, 지금까지 표 102개, 대기 중인 카테고리 15개

중단됨 - 지금까지 모은 표 138개는 kosis_full_catalog_tf.csv 에 저장합니다.
중단됨. 그동안 저장된 kosis_full_catalog_tf.csv 을 그대로 쓰시면 됩니다.
이어서 더 모으고 싶으면 --resume 옵션으로 같은 --output 을 다시 실행하세요 (이미 모은 표는 중복 없이, 카테고리는 처음부터 다시 훑지만 표 자체는 안 겹칩니다).


In [32]:
import pandas as pd

df_meta_combined = pd.concat([
    pd.read_csv("kosis_meta_index_v2.csv"),   # 재정+도소매+소득소비+물가 (10065개)
    pd.read_csv("kosis_meta_index_tf.csv"),   # 무역+금융 (2754개)
], ignore_index=True).drop_duplicates()
df_meta_combined.to_csv("kosis_meta_index_final.csv", index=False, encoding="utf-8-sig")

df_summary_combined = pd.concat([
    pd.read_csv("kosis_metadata_summary_enriched.csv"),
    pd.read_csv("kosis_metadata_summary_enriched_tf.csv"),
], ignore_index=True).drop_duplicates(subset=["ORG_ID", "TBL_ID"])
df_summary_combined.to_csv("kosis_metadata_summary_enriched_final.csv", index=False, encoding="utf-8-sig")

print("meta:", len(df_meta_combined), "| summary:", len(df_summary_combined))

meta: 179862 | summary: 4812


In [34]:
import pandas as pd

df = pd.read_csv("kosis_metadata_summary_enriched_final.csv")
df["top_category"] = df["category_path"].str.split(" > ").str[0]
print(df["top_category"].value_counts())

top_category
금융         2443
도소매ㆍ서비스    1988
무역ㆍ국제수지     311
정부ㆍ재정        70
Name: count, dtype: int64


In [35]:
df_validated.groupby("metric_domain")["mapping_status"].value_counts()

metric_domain  mapping_status    
금융·금리          MAPPING_FAILED         39
               NOT_EVALUATED          20
               NEEDS_CONFIRMATION      1
무역             MAPPING_FAILED         83
               NOT_EVALUATED          48
               NEEDS_CONFIRMATION     13
물가             MAPPING_FAILED         10
               NOT_EVALUATED           6
               NEEDS_CONFIRMATION      2
보건·복지          MAPPING_FAILED          2
               NOT_EVALUATED           1
소득·임금          MAPPING_FAILED          2
               NOT_EVALUATED           1
소매·소비          MAPPING_FAILED        108
               NOT_EVALUATED          55
               NEEDS_CONFIRMATION      2
인구             MAPPING_FAILED          8
               NOT_EVALUATED           4
재정·조세          MAPPING_FAILED         12
               NOT_EVALUATED           7
               NEEDS_CONFIRMATION      2
Name: count, dtype: int64

In [36]:
import pandas as pd
df = pd.read_csv("kosis_validated_final.csv")

# 1순위 후보 중 NEEDS_CONFIRMATION만, 거리 낮은(그나마 유력한) 순으로
rank1_review = df[(df["candidate_rank"] == 1) & (df["mapping_status"] == "NEEDS_CONFIRMATION")].copy()
rank1_review["cosine_distance"] = rank1_review["match_reason"].str.extract(r"cosine_distance=([\d.]+)").astype(float)
rank1_review = rank1_review.sort_values("cosine_distance")

cols = ["claim_measurement_id", "metric_domain", "indicator", "value", "unit", "tbl_nm", "cosine_distance"]
rank1_review[cols].to_csv("review_priority.csv", index=False, encoding="utf-8-sig")
print(len(rank1_review), "건 - 거리 낮은 순으로 review_priority.csv 에 정렬해뒀습니다")
rank1_review[cols].head(20)

13 건 - 거리 낮은 순으로 review_priority.csv 에 정렬해뒀습니다


,claim_measurement_id,metric_domain,indicator,value,unit,tbl_nm,cosine_distance
153,B0020-s005-m1,무역,수출액 차이,5.235000e+09,달러,"국가별 수출액, 수입액",0.2802
147,B0020-s004-m1,무역,대미 수출액,1.277910e+11,달러,"국가별 수출액, 수입액",0.3126
141,B0020-s003-m1,무역,대중 수출액,1.330260e+11,달러,"국가별 수출액, 수입액",0.3173
156,B0020-s006-m1,무역,대중 수출액과 대미 수출액 간의 격차,8.900000e+08,달러,"국가별 수출액, 수입액",0.3301
159,B0020-s007-m2,무역,대중 수출액과 대미 수출액 간의 격차,8.940500e+10,달러,"국가별 수출액, 수입액",0.3324
162,B0020-s008-m2,무역,대미 수출액,1.000000e+11,달러,"국가별 수출액, 수입액",0.3366
165,B0020-s010-m1,무역,대중 수출액,1.168380e+11,달러,"국가별 수출액, 수입액",0.3419
168,B0020-s010-m3,무역,대중 수출액,1.629130e+11,달러,"국가별 수출액, 수입액",0.3419
387,B0020-s010-m2,무역,대중 수출액,1.000000e+11,달러,"국가별 수출액, 수입액",0.3419
423,B0048-s027-m1,금융·금리,원화 환율,1.453500e+03,원,한국은행 원화대출금,0.3700


In [37]:
import pandas as pd
df = pd.read_csv("kosis_validated_final.csv")
rank1_review = df[(df["candidate_rank"] == 1) & (df["mapping_status"] == "NEEDS_CONFIRMATION")].copy()
rank1_review["cosine_distance"] = rank1_review["match_reason"].str.extract(r"cosine_distance=([\d.]+)").astype(float)

# 도메인별로 거리 가장 낮은(가장 유력한) 것 3개씩만
cols = ["claim_measurement_id", "indicator", "value", "unit", "tbl_nm", "cosine_distance"]
for domain, group in rank1_review.groupby("metric_domain"):
    print(f"=== {domain} ===")
    print(group.sort_values("cosine_distance")[cols].head(3).to_string(index=False))
    print()

=== 금융·금리 ===
claim_measurement_id indicator  value unit     tbl_nm  cosine_distance
       B0048-s027-m1     원화 환율 1453.5    원 한국은행 원화대출금             0.37

=== 무역 ===
claim_measurement_id indicator        value unit       tbl_nm  cosine_distance
       B0020-s005-m1    수출액 차이 5.235000e+09   달러 국가별 수출액, 수입액           0.2802
       B0020-s004-m1    대미 수출액 1.277910e+11   달러 국가별 수출액, 수입액           0.3126
       B0020-s003-m1    대중 수출액 1.330260e+11   달러 국가별 수출액, 수입액           0.3173

=== 소매·소비 ===
claim_measurement_id indicator        value unit       tbl_nm  cosine_distance
       B0011-s007-m1      영업이익 1.979420e+13    원 코스닥 상장사 수익현황           0.4097
       B0011-s006-m1      영업이익 3.965310e+13    원 코스닥 상장사 수익현황           0.4097

=== 재정·조세 ===
claim_measurement_id indicator        value unit     tbl_nm  cosine_distance
       B0046-s005-m1   순자금 운용액 2.980000e+13    원 가계대출 잔액 비중           0.4176



In [38]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
